In [1]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".projectroot").exists())

# Master Join v1 : country-year risk-model table

This notebook joins the cleaned source tables into one country-year table, the feature set the risk model trains on.

**Input:** the cleaned per-source tables in `data/interim/` (wdi, vdem, gdelt, gsdb, sipri, ungavoting, unhcr, comtrade) and the GPR target in `data/interim/gpr/`.  
**Output:** `data/processed/master_v1.csv` (one row per country-year, features plus the GPR target).

In [2]:
import os, glob
import numpy as np, pandas as pd
C = str(ROOT / "data/interim")
OUT_DIR = str(ROOT / "data/processed"); os.makedirs(OUT_DIR, exist_ok=True)
Y0, Y1 = 2015, 2024
def load(name): return pd.read_csv(sorted(glob.glob(f'{C}/{name}/*.csv'))[0])

## 1. Annual feature sources (2015-2024)

In [3]:
wdi   = load('worldbank')
vdem  = load('vdem')
gsdb  = load('gsdb')
unga  = load('ungavoting')
unhcr = load('unhcr')
comt  = load('comtrade')
sipri = load('sipri')
annual = {'wdi':wdi,'vdem':vdem,'gsdb':gsdb,'unga':unga,'unhcr':unhcr,'comtrade':comt,'sipri':sipri}
for k,v in annual.items():
    annual[k] = v[(v.year>=Y0)&(v.year<=Y1)]
    print(f'{k:9s} {annual[k].shape} | countries {annual[k].iso3.nunique()}')

wdi       (2170, 20) | countries 217
vdem      (1790, 7) | countries 179
gsdb      (1154, 11) | countries 156
unga      (1925, 6) | countries 193
unhcr     (2027, 6) | countries 211
comtrade  (1690, 22) | countries 188
sipri     (1269, 5) | countries 172


## 2. GDELT country-month -> annual (volume summed, shares/means event-weighted)

In [4]:
gd = load('gdelt')
gd['year'] = pd.to_datetime(gd['month']).dt.year
gd = gd[(gd.year>=Y0)&(gd.year<=Y1)]
wcols = ['gdelt_avg_goldstein','gdelt_avg_tone','gdelt_material_conflict_share','gdelt_violence_share','gdelt_protest_share','gdelt_threat_share']
def roll(g):
    w = g['gdelt_event_count'].clip(lower=0); tot = w.sum()
    out = {'gdelt_event_count': g['gdelt_event_count'].sum()}
    for c in wcols: out[c] = np.average(g[c], weights=w) if tot>0 else np.nan
    bw = g['bilateral_conflict_events'].fillna(0); bt = bw.sum()
    out['gdelt_adversary_concentration'] = (np.average(g['gdelt_adversary_concentration'].fillna(0), weights=bw) if bt>0 else np.nan)
    return pd.Series(out)
_use = ['gdelt_event_count'] + wcols + ['bilateral_conflict_events','gdelt_adversary_concentration']
gdelt_y = gd.groupby(['iso3','year'])[_use].apply(roll).reset_index()
print('gdelt annual:', gdelt_y.shape, '| countries', gdelt_y.iso3.nunique())

gdelt annual: (2388, 10) | countries 239


## 3. GPR target (now long: iso3, month, gpr -> annual mean)

In [5]:
gpr = pd.read_csv(f'{C}/gpr/gpr_monthly.csv')
gpr['year'] = pd.to_datetime(gpr['month']).dt.year
target = (gpr[(gpr.year>=Y0)&(gpr.year<=Y1)].groupby(['iso3','year'])['gpr'].mean().reset_index())
print('target rows:', target.shape, '| GPR countries', target.iso3.nunique())

target rows: (440, 3) | GPR countries 44


## 4. Build spine (union of countries x years) and left-join everything

In [6]:
isos = set()
for v in annual.values(): isos |= set(v.iso3)
isos |= set(gdelt_y.iso3)
isos = sorted(i for i in isos if isinstance(i,str) and len(i)==3)
spine = pd.MultiIndex.from_product([isos, range(Y0,Y1+1)], names=['iso3','year']).to_frame(index=False)
m = spine.copy()
for k,v in annual.items(): m = m.merge(v, on=['iso3','year'], how='left')
m = m.merge(gdelt_y, on=['iso3','year'], how='left')
m = m.merge(target,  on=['iso3','year'], how='left')
print('spine:', spine.shape, '| joined:', m.shape)

spine: (2470, 2) | joined: (2470, 74)


## 5. Apply fill rules

In [7]:
zero_fill = (['n_senders','n_cases','sanc_multilateral','sanc_arms','sanc_military','sanc_trade','sanc_financial','sanc_travel','sanc_other']
             + ['refugees_origin','idps','refugees_hosted','stateless']
             + ['arms_imports_tiv','arms_exports_tiv','arms_n_suppliers']
             + ['gdelt_event_count'])
zero_fill = [c for c in zero_fill if c in m.columns]
m[zero_fill] = m[zero_fill].fillna(0)
print('zero-filled:', len(zero_fill), 'cols | predict-only (NaN target) rows:', int(m.gpr.isna().sum()))

zero-filled: 17 cols | predict-only (NaN target) rows: 2030


## 6. Verify + save

In [8]:
meta = ['iso3','year','country','region','income_group']; target_col=['gpr']
feat = [c for c in m.columns if c not in meta+target_col]
m = m[[c for c in meta if c in m.columns]+target_col+feat]
assert m.duplicated(['iso3','year']).sum()==0, 'duplicate keys!'
print('MASTER:', m.shape, '| countries', m.iso3.nunique(), '| years', m.year.min(),'-',m.year.max())
print('labeled (GPR) country-years:', int(m.gpr.notna().sum()), '| labeled countries:', m.loc[m.gpr.notna(),'iso3'].nunique())
probe = {'wdi':'gdp_per_capita','vdem':'lib_democracy','gsdb':'n_cases','unga':'ideal_point','unhcr':'refugees_origin','comtrade':'total_trade_usd','sipri':'arms_imports_tiv','gdelt':'gdelt_event_count'}
print('\nper-source non-null coverage (% of rows):')
for k,col in probe.items(): print(f'  {k:9s} {col:22s} {100*m[col].notna().mean():.0f}%')
m.to_csv(f'{OUT_DIR}/master_v1.csv', index=False)
print('\nsaved:', f'{OUT_DIR}/master_v1.csv')

MASTER: (2470, 74) | countries 247 | years 2015 - 2024
labeled (GPR) country-years: 440 | labeled countries: 44

per-source non-null coverage (% of rows):
  wdi       gdp_per_capita         84%
  vdem      lib_democracy          72%
  gsdb      n_cases                100%
  unga      ideal_point            78%
  unhcr     refugees_origin        100%
  comtrade  total_trade_usd        68%
  sipri     arms_imports_tiv       100%
  gdelt     gdelt_event_count      100%

saved: /Users/oussamaennaciri/Documents/Education/Academique/Regis/MSDS692 S40 Data Science Practicum/data/processed/master_v1.csv


Claude (Anthropic) was used only as a collaborator for writing code and polishing the notebooks. All analytical decisions, interpretations, and research were conducted independently by me.